In [ ]:
import pandas as pd
import conexion_api


In [ ]:
url = "https://servicios.ine.es/wstempus/js/ES/DATOS_TABLA/76136?nlast=60&det=2"

In [ ]:
data = conexion_api.llamada_api(url)

In [ ]:
len(data)

In [ ]:
data

In [ ]:
data[70]['Nombre']    #Ruta al nombre

In [ ]:
data[0]['Data'][0]['Periodo']['Mes_inicio']     #Ruta al mes

In [ ]:
data[0]['Data'][0]['Anyo']       #Ruta al año

In [ ]:
data[0]['Data'][0]['Periodo']['Nombre_largo']       #Ruta al nombre del mes

In [ ]:
data[0]['Data'][0]['Valor']             #Ruta al IPC

In [ ]:
data[0]['Data']

In [ ]:
for serie in data:
    for dato in serie['Data']:
        print(f'{'=*' * 50}\n{serie['Nombre']}\n{'=' * 100}\n- Numero Mes: {dato['Periodo']['Mes_inicio']}\n{'=' * 100}\n- Numero Año: {dato['Anyo']}\n{'=' * 100}\n- Mes: {dato['Periodo']['Nombre_largo']}\n{'=' * 100}\n- IPC: {dato['Valor']}\n{'=' * 100}\n-Codigo fecha: {dato['CodigoPeriodo']}\n{'=' * 50}\n')

In [ ]:
tiempo = {'id_tiempo': [], 'anio': [], 'mes': [], 'nombre_mes': [] }

for serie in data:
    for dato in serie['Data']:
        if dato['CodigoPeriodo'] not in tiempo['id_tiempo']:
            tiempo['id_tiempo'].append(dato['CodigoPeriodo'])
            tiempo['anio'].append(dato['Anyo'])
            tiempo['mes'].append(dato['Periodo']['Mes_inicio'])
            tiempo['nombre_mes'].append(dato['Periodo']['Nombre_largo'])


In [ ]:
tiempo = pd.DataFrame(tiempo)
tiempo.sample()

In [ ]:
tiempo.shape

In [ ]:
tiempo.to_csv('tiempo.csv', index=False)

In [ ]:
territorio = {'id_territorio': [], 'nombre_territorio': []}
id_ter = 1

for serie in data:
    nombre_completo = serie['Nombre']
    nombre_limpio = nombre_completo.split('.')[0].strip()

    if nombre_limpio not in territorio['nombre_territorio']:
        territorio['id_territorio'].append(id_ter)
        territorio['nombre_territorio'].append(nombre_limpio)

        id_ter += 1

In [ ]:
territorio = pd.DataFrame(territorio)
territorio.sample()

In [ ]:
territorio.shape

In [ ]:
territorio.to_csv('territorio.csv', index=False)

In [ ]:
sectores_ipc = {'id_sector': [], 'nombre_sector': []}
id_sec = 1

for serie in data:
    nombre_completo = serie['Nombre']
    sector_limpio = nombre_completo.split('.')[1].strip()

    if sector_limpio not in sectores_ipc['nombre_sector']:
        sectores_ipc['id_sector'].append(id_sec)
        sectores_ipc['nombre_sector'].append(sector_limpio)

        id_sec += 1        

In [ ]:
sectores_ipc = pd.DataFrame(sectores_ipc)

In [ ]:
sectores_ipc

In [ ]:
sectores_ipc.to_csv('sectores_ipc.csv', index=False)

In [ ]:
tipo_medida = {'id_medida': [], 'nombre_medida': []}
id_med = 1

for serie in data:
    nombre_completo = serie['Nombre']
    medida_limpio = nombre_completo.split('.')[2].strip()

    if medida_limpio not in tipo_medida['nombre_medida']:
        tipo_medida['id_medida'].append(id_med)
        tipo_medida['nombre_medida'].append(medida_limpio)

        id_med += 1     

In [ ]:
tipo_medida

In [ ]:
tipo_medida = pd.DataFrame(tipo_medida)

In [ ]:
tipo_medida

In [ ]:
tipo_medida.to_csv('tipo_medida.csv', index=False)

In [ ]:
# =============================================================================

# 1. PREPARACIÓN: Creamos diccionarios de búsqueda rápida (Mapeos)

# =============================================================================

# Esto sirve para que Python asocie al vuelo un nombre con su ID.

# Ejemplo: map_territorios['Andalucía'] nos devolverá directamente el ID 2.
 
map_territorios = dict(zip(territorio['nombre_territorio'], territorio['id_territorio']))

map_sectores = dict(zip(sectores_ipc['nombre_sector'], sectores_ipc['id_sector']))

map_medidas = dict(zip(tipo_medida['nombre_medida'], tipo_medida['id_medida']))
 
# Para el tiempo, como vuestra clave primaria es el 'id_tiempo' (ej: '202512'), 

# podemos usar directamente el 'CodigoPeriodo' que viene en la API como ID.
 
# =============================================================================

# 2. ESTRUCTURA DE LA TABLA DE HECHOS

# =============================================================================

ipc = {

    'id_tiempo': [],

    'id_territorio': [],

    'id_sector': [],

    'id_medida': [],

    'valor_ipc': []

}
 
# =============================================================================

# 3. EL BUCLE PRINCIPAL (Extracción y Cruce de IDs)

# =============================================================================

for serie in data:

    nombre_completo = serie['Nombre']

    partes = nombre_completo.split('.')

    if len(partes) >= 3:

        # Extraemos los nombres de texto limpio

        territorio_texto = partes[0].strip()

        sector_texto = partes[1].strip()

        medida_texto = partes[2].strip()

        # Corregimos si dice 'Nacional' para que coincida con vuestra dimensión

        if territorio_texto.lower() == 'nacional':

            territorio_texto = 'Total Nacional'

        # BUSQUEDA DE IDs: Miramos en nuestros mapas qué ID le toca a cada texto

        id_terr = map_territorios.get(territorio_texto)

        id_sec = map_sectores.get(sector_texto)

        id_med = map_medidas.get(medida_texto)

        # Si por algún motivo el INE da un texto que no guardamos antes, nos lo saltamos

        if id_terr is None or id_sec is None or id_med is None:

            continue

        # Ahora bajamos al segundo nivel: los datos históricos de esta serie

        for dato in serie['Data']:

            # Pescamos el valor numérico (la métrica)

            valor = dato.get('Valor')

            # Sacamos el código del periodo para usarlo como ID de tiempo (ej: '202605')

            id_time = dato.get('CodigoPeriodo')

            # Si tenemos todos los datos, ¡hacemos el append a la tabla de hechos!

            if valor is not None and id_time is not None:

                ipc['id_tiempo'].append(str(id_time))

                ipc['id_territorio'].append(id_terr)

                ipc['id_sector'].append(id_sec)

                ipc['id_medida'].append(id_med)

                ipc['valor_ipc'].append(valor)
 
 

In [ ]:
ipc

In [ ]:
ipc = pd.DataFrame(ipc)

ipc.sample(5)

In [ ]:
ipc.to_csv('ipc.csv', index=False)